In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('../../../fraud_oracle.csv')
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,21,Policy Holder,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,34,Policy Holder,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,47,Policy Holder,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,65,Third Party,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,27,Third Party,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


In [ ]:
# Fraud Oracle: Insurance Claim Fraud Modeling

This notebook uses the `fraud_oracle.csv` dataset to build a clean baseline fraud-risk model for the RiskProof AI project. The target is `FraudFound_P`, and the notebook evaluates both a single-feature linear regression baseline and a multivariate linear regression baseline.

,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level
0,26.0,12.0,10.33,4.45,2.0,11.0,4.0,1.0,15.07,0.14,55.96,Medium
1,39.0,10.0,8.62,5.77,5.0,15.0,11.0,5.0,13.25,0.54,82.22,High
2,34.0,13.0,NaN,4.03,5.0,2.0,18.0,9.0,11.18,1.54,61.77,Medium
3,30.0,1.0,6.85,6.47,2.0,15.0,26.0,1.0,11.14,0.96,54.98,Medium
4,27.0,7.0,4.24,5.80,NaN,9.0,17.0,7.0,8.05,0.36,27.90,Low


In [3]:
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())
print('\nDataset info:')
df.info()
print('\nMissing values per column:')
display(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nTarget distribution:')
display(df['FraudFound_P'].value_counts())

Dataset shape: (15420, 33)

First 5 rows:


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,21,Policy Holder,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,34,Policy Holder,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,47,Policy Holder,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,65,Third Party,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,27,Third Party,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision



Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Month                 15420 non-null  str  
 1   WeekOfMonth           15420 non-null  int64
 2   DayOfWeek             15420 non-null  str  
 3   Make                  15420 non-null  str  
 4   AccidentArea          15420 non-null  str  
 5   DayOfWeekClaimed      15420 non-null  str  
 6   MonthClaimed          15420 non-null  str  
 7   WeekOfMonthClaimed    15420 non-null  int64
 8   Sex                   15420 non-null  str  
 9   MaritalStatus         15420 non-null  str  
 10  Age                   15420 non-null  int64
 11  Fault                 15420 non-null  str  
 12  PolicyType            15420 non-null  str  
 13  VehicleCategory       15420 non-null  str  
 14  VehiclePrice          15420 non-null  str  
 15  FraudFound_P          15420 non-null  int64
 16  

Month                   0
WeekOfMonth             0
DayOfWeek               0
Make                    0
AccidentArea            0
DayOfWeekClaimed        0
MonthClaimed            0
WeekOfMonthClaimed      0
Sex                     0
MaritalStatus           0
Age                     0
Fault                   0
PolicyType              0
VehicleCategory         0
VehiclePrice            0
FraudFound_P            0
PolicyNumber            0
RepNumber               0
Deductible              0
DriverRating            0
Days_Policy_Accident    0
Days_Policy_Claim       0
PastNumberOfClaims      0
AgeOfVehicle            0
AgeOfPolicyHolder       0
PoliceReportFiled       0
WitnessPresent          0
AgentType               0
NumberOfSuppliments     0
AddressChange_Claim     0
NumberOfCars            0
Year                    0
BasePolicy              0
dtype: int64


Duplicate rows: 0

Target distribution:


FraudFound_P
0    14497
1      923
Name: count, dtype: int64

In [4]:
target_col = 'FraudFound_P'

# Standardize column names and text values
cleaned_df = df.copy()
cleaned_df.columns = cleaned_df.columns.str.strip()
for column in cleaned_df.select_dtypes(include=['object', 'string']).columns:
    cleaned_df[column] = cleaned_df[column].astype(str).str.strip()

# Treat common placeholders as missing values
cleaned_df = cleaned_df.replace({'': np.nan, 'NA': np.nan, 'N/A': np.nan, 'null': np.nan, 'None': np.nan})

# Remove duplicate rows
cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)

# Fill missing values column-wise
for column in cleaned_df.columns:
    if column == target_col:
        continue
    if cleaned_df[column].dtype.kind in 'biufc':
        cleaned_df[column] = cleaned_df[column].fillna(cleaned_df[column].median())
    else:
        cleaned_df[column] = cleaned_df[column].fillna(cleaned_df[column].mode()[0])

# Final safety checks
print('Cleaned shape:', cleaned_df.shape)
print('Remaining missing values:', int(cleaned_df.isna().sum().sum()))
print('Remaining duplicates:', int(cleaned_df.duplicated().sum()))
display(cleaned_df.head())

C:\Users\UTHANDAM\AppData\Local\Temp\ipykernel_16172\1167932582.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in cleaned_df.select_dtypes(include='object').columns:


Cleaned shape: (15420, 33)
Remaining missing values: 0
Remaining duplicates: 0


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,21,Policy Holder,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,34,Policy Holder,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,47,Policy Holder,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,65,Third Party,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,27,Third Party,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


In [5]:
base_df = cleaned_df.drop(columns=['PolicyNumber'])
print('Modeling dataframe shape:', base_df.shape)
print('\nTarget distribution after cleaning:')
display(base_df[target_col].value_counts(normalize=True).rename('fraction'))

numeric_columns = base_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = base_df.select_dtypes(exclude=[np.number]).columns.tolist()
print('\nNumeric columns:', numeric_columns)
print('\nCategorical columns:', categorical_columns)

Modeling dataframe shape: (15420, 32)

Target distribution after cleaning:


FraudFound_P
0    0.940143
1    0.059857
Name: fraction, dtype: float64


Numeric columns: ['WeekOfMonth', 'WeekOfMonthClaimed', 'Age', 'FraudFound_P', 'RepNumber', 'Deductible', 'DriverRating', 'Year']

Categorical columns: ['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'BasePolicy']


In [9]:
def regression_metrics(y_true, y_pred):
    predicted_class = np.clip(np.rint(y_pred), 0, 1).astype(int)
    mse = mean_squared_error(y_true, y_pred)
    return {
        'accuracy': accuracy_score(y_true, predicted_class),
        'mae': mean_absolute_error(y_true, y_pred),
        'mse': mse,
        'rmse': float(np.sqrt(mse)),
        'r2': r2_score(y_true, y_pred),
    }

single_feature_candidates = [column for column in numeric_columns if column != target_col]
single_feature_candidates

['WeekOfMonth',
 'WeekOfMonthClaimed',
 'Age',
 'RepNumber',
 'Deductible',
 'DriverRating',
 'Year']

In [10]:
single_results = []
search_test_sizes = [0.2, 0.25, 0.3]
search_random_states = [0, 42, 101]

for feature in single_feature_candidates:
    X = base_df[[feature]]
    y = base_df[target_col]
    for test_size in search_test_sizes:
        for random_state in search_random_states:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=random_state, stratify=y
            )
            model = LinearRegression()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            metrics = regression_metrics(y_test, y_pred)
            single_results.append({
                'feature': feature,
                'test_size': test_size,
                'random_state': random_state,
                **metrics,
            })

single_results_df = pd.DataFrame(single_results).sort_values(['accuracy', 'r2', 'mae'], ascending=[False, False, True])
display(single_results_df.head(10))

,feature,test_size,random_state,accuracy,mae,mse,rmse,r2
26,Age,0.3,101,0.940121,0.112469,0.056201,0.237067,0.001646
25,Age,0.3,42,0.940121,0.112416,0.056207,0.237080,0.001540
24,Age,0.3,0,0.940121,0.112325,0.056236,0.237142,0.001018
61,Year,0.3,42,0.940121,0.112216,0.056265,0.237203,0.000503
43,Deductible,0.3,42,0.940121,0.112376,0.056277,0.237227,0.000295
62,Year,0.3,101,0.940121,0.112503,0.056281,0.237237,0.000217
42,Deductible,0.3,0,0.940121,0.112479,0.056287,0.237250,0.000107
7,WeekOfMonth,0.3,42,0.940121,0.112541,0.056288,0.237250,0.000101
51,DriverRating,0.3,0,0.940121,0.112580,0.056288,0.237251,0.000100
16,WeekOfMonthClaimed,0.3,42,0.940121,0.112577,0.056290,0.237255,0.000063


In [11]:
best_single = single_results_df.iloc[0]
best_single_feature = best_single['feature']
best_single_test_size = float(best_single['test_size'])
best_single_random_state = int(best_single['random_state'])

X_single = base_df[[best_single_feature]]
y_single = base_df[target_col]
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_single,
    y_single,
    test_size=best_single_test_size,
    random_state=best_single_random_state,
    stratify=y_single,
)

single_model = LinearRegression()
single_model.fit(X_train_s, y_train_s)
single_pred = single_model.predict(X_test_s)
single_metrics = regression_metrics(y_test_s, single_pred)

print('Best single-feature model')
print('Feature:', best_single_feature)
print('Test size:', best_single_test_size)
print('Random state:', best_single_random_state)
print('Metrics:', single_metrics)

sample_single = pd.DataFrame({best_single_feature: X_test_s.iloc[:5, 0].values})
print('\nSample predictions:')
for actual, pred in zip(y_test_s.iloc[:5].values, single_model.predict(sample_single)):
    print(f'Actual={actual}  Predicted_score={pred:.4f}  Predicted_label={int(np.clip(np.rint(pred), 0, 1))}')

Best single-feature model
Feature: Age
Test size: 0.3
Random state: 101
Metrics: {'accuracy': 0.9401210549070471, 'mae': 0.11246895093167834, 'mse': 0.05620077016894841, 'rmse': 0.23706701619784312, 'r2': 0.0016464943432824564}

Sample predictions:
Actual=0  Predicted_score=0.0583  Predicted_label=0
Actual=0  Predicted_score=0.0609  Predicted_label=0
Actual=0  Predicted_score=0.0624  Predicted_label=0
Actual=0  Predicted_score=0.0572  Predicted_label=0
Actual=0  Predicted_score=0.0470  Predicted_label=0


## Multivariate Linear Regression

Now we use all available features together with one-hot encoding to build a multivariate linear regression baseline and compare it against the best single-feature model.

In [12]:
X_multi = pd.get_dummies(base_df.drop(columns=[target_col]), drop_first=True)
y_multi = base_df[target_col]

print('Encoded multivariate feature matrix shape:', X_multi.shape)
display(X_multi.head())

Encoded multivariate feature matrix shape: (15420, 123)


,WeekOfMonth,WeekOfMonthClaimed,Age,RepNumber,Deductible,DriverRating,Year,Month_Aug,Month_Dec,Month_Feb,Month_Jan,Month_Jul,Month_Jun,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,DayOfWeek_Monday,DayOfWeek_Saturday,DayOfWeek_Sunday,DayOfWeek_Thursday,DayOfWeek_Tuesday,DayOfWeek_Wednesday,Make_BMW,Make_Chevrolet,Make_Dodge,Make_Ferrari,Make_Ford,Make_Honda,Make_Jaguar,Make_Lexus,Make_Mazda,Make_Mecedes,Make_Mercury,Make_Nisson,Make_Pontiac,Make_Porche,Make_Saab,Make_Saturn,Make_Toyota,Make_VW,AccidentArea_Urban,DayOfWeekClaimed_Friday,DayOfWeekClaimed_Monday,DayOfWeekClaimed_Saturday,DayOfWeekClaimed_Sunday,DayOfWeekClaimed_Thursday,DayOfWeekClaimed_Tuesday,DayOfWeekClaimed_Wednesday,MonthClaimed_Apr,MonthClaimed_Aug,MonthClaimed_Dec,MonthClaimed_Feb,MonthClaimed_Jan,MonthClaimed_Jul,MonthClaimed_Jun,MonthClaimed_Mar,MonthClaimed_May,MonthClaimed_Nov,MonthClaimed_Oct,MonthClaimed_Sep,Sex_Male,MaritalStatus_Married,MaritalStatus_Single,MaritalStatus_Widow,Fault_Third Party,PolicyType_Sedan - Collision,PolicyType_Sedan - Liability,PolicyType_Sport - All Perils,PolicyType_Sport - Collision,PolicyType_Sport - Liability,PolicyType_Utility - All Perils,PolicyType_Utility - Collision,PolicyType_Utility - Liability,VehicleCategory_Sport,VehicleCategory_Utility,VehiclePrice_30000 to 39000,VehiclePrice_40000 to 59000,VehiclePrice_60000 to 69000,VehiclePrice_less than 20000,VehiclePrice_more than 69000,Days_Policy_Accident_15 to 30,Days_Policy_Accident_8 to 15,Days_Policy_Accident_more than 30,Days_Policy_Accident_none,Days_Policy_Claim_8 to 15,Days_Policy_Claim_more than 30,Days_Policy_Claim_none,PastNumberOfClaims_2 to 4,PastNumberOfClaims_more than 4,PastNumberOfClaims_none,AgeOfVehicle_3 years,AgeOfVehicle_4 years,AgeOfVehicle_5 years,AgeOfVehicle_6 years,AgeOfVehicle_7 years,AgeOfVehicle_more than 7,AgeOfVehicle_new,AgeOfPolicyHolder_18 to 20,AgeOfPolicyHolder_21 to 25,AgeOfPolicyHolder_26 to 30,AgeOfPolicyHolder_31 to 35,AgeOfPolicyHolder_36 to 40,AgeOfPolicyHolder_41 to 50,AgeOfPolicyHolder_51 to 65,AgeOfPolicyHolder_over 65,PoliceReportFiled_Yes,WitnessPresent_Yes,AgentType_Internal,NumberOfSuppliments_3 to 5,NumberOfSuppliments_more than 5,NumberOfSuppliments_none,AddressChange_Claim_2 to 3 years,AddressChange_Claim_4 to 8 years,AddressChange_Claim_no change,AddressChange_Claim_under 6 months,NumberOfCars_2 vehicles,NumberOfCars_3 to 4,NumberOfCars_5 to 8,NumberOfCars_more than 8,BasePolicy_Collision,BasePolicy_Liability
0,5,1,21,12,300,1,1994,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,True,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True
1,3,4,34,15,400,4,1994,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False
2,5,2,47,7,400,3,1994,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,Fal

In [13]:
multi_results = []

for test_size in search_test_sizes:
    for random_state in search_random_states:
        X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
            X_multi,
            y_multi,
            test_size=test_size,
            random_state=random_state,
            stratify=y_multi,
        )
        multi_model = LinearRegression()
        multi_model.fit(X_train_m, y_train_m)
        y_pred_m = multi_model.predict(X_test_m)
        metrics = regression_metrics(y_test_m, y_pred_m)
        multi_results.append({
            'test_size': test_size,
            'random_state': random_state,
            **metrics,
        })

multi_results_df = pd.DataFrame(multi_results).sort_values(['accuracy', 'r2', 'mae'], ascending=[False, False, True])
display(multi_results_df.head(10))

,test_size,random_state,accuracy,mae,mse,rmse,r2
3,0.25,0,0.940337,0.116032,0.052489,0.229106,0.068205
6,0.30,0,0.940337,0.115522,0.052599,0.229344,0.065632
7,0.30,42,0.940121,0.116485,0.052877,0.229950,0.060686
8,0.30,101,0.940121,0.117023,0.053115,0.230466,0.056468
5,0.25,101,0.940078,0.116749,0.053083,0.230397,0.057675
4,0.25,42,0.940078,0.116741,0.053237,0.230732,0.054927
1,0.20,42,0.940013,0.115974,0.052709,0.229585,0.065251
0,0.20,0,0.940013,0.116974,0.052969,0.230151,0.060635
2,0.20,101,0.940013,0.117192,0.053347,0.230970,0.053935


## Model Comparison and Conclusion

The target is `FraudFound_P`.

- `0` means the claim is not fraud
- `1` means the claim is fraud

The most useful output for RiskProof AI is a fraud score, a predicted fraud label, and the strongest claim-risk drivers so brokers and insurers can review the claim faster.

In [14]:
best_multi = multi_results_df.iloc[0]
best_multi_test_size = float(best_multi['test_size'])
best_multi_random_state = int(best_multi['random_state'])

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi,
    y_multi,
    test_size=best_multi_test_size,
    random_state=best_multi_random_state,
    stratify=y_multi,
)

final_multi_model = LinearRegression()
final_multi_model.fit(X_train_m, y_train_m)
multi_pred = final_multi_model.predict(X_test_m)
multi_metrics = regression_metrics(y_test_m, multi_pred)

print('Best multivariate model')
print('Test size:', best_multi_test_size)
print('Random state:', best_multi_random_state)
print('Metrics:', multi_metrics)

coef_df = pd.DataFrame({
    'feature': X_multi.columns,
    'coefficient': final_multi_model.coef_,
})
coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
print('\nTop 15 coefficients by magnitude:')
display(coef_df.sort_values('abs_coefficient', ascending=False).head(15))

Best multivariate model
Test size: 0.25
Random state: 0
Metrics: {'accuracy': 0.940337224383917, 'mae': 0.1160316901387239, 'mse': 0.05248943792723756, 'rmse': 0.22910573525609865, 'r2': 0.06820469444608612}

Top 15 coefficients by magnitude:


,feature,coefficient,abs_coefficient
116,AddressChange_Claim_under 6 months,0.845168,0.845168
27,Make_Ferrari,-0.177961,0.177961
30,Make_Jaguar,-0.135295,0.135295
31,Make_Lexus,-0.128071,0.128071
33,Make_Mecedes,0.125896,0.125896
37,Make_Porche,-0.108927,0.108927
66,Fault_Third Party,-0.093402,0.093402
120,NumberOfCars_more than 8,-0.091490,0.091490
41,Make_VW,-0.086162,0.086162
113,AddressChange_Claim_2 to 3 years,0.083455,0.083455


In [15]:
comparison_df = pd.DataFrame([
    {
        'model': f'Single feature ({best_single_feature})',
        'accuracy': single_metrics['accuracy'],
        'mae': single_metrics['mae'],
        'rmse': single_metrics['rmse'],
        'r2': single_metrics['r2'],
    },
    {
        'model': 'Multivariate linear regression',
        'accuracy': multi_metrics['accuracy'],
        'mae': multi_metrics['mae'],
        'rmse': multi_metrics['rmse'],
        'r2': multi_metrics['r2'],
    },
])

display(comparison_df.sort_values(['accuracy', 'r2', 'mae'], ascending=[False, False, True]))

best_row = comparison_df.sort_values(['accuracy', 'r2', 'mae'], ascending=[False, False, True]).iloc[0]
print(f"Best baseline model: {best_row['model']}")
print(f"Accuracy: {best_row['accuracy']:.4f}")
print(f"RMSE: {best_row['rmse']:.4f}")

,model,accuracy,mae,rmse,r2
1,Multivariate linear regression,0.940337,0.116032,0.229106,0.068205
0,Single feature (Age),0.940121,0.112469,0.237067,0.001646


Best baseline model: Multivariate linear regression
Accuracy: 0.9403
RMSE: 0.2291


In [ ]:
## Final Output and Usefulness

This notebook gives you a baseline fraud-risk model for the `fraud_oracle` dataset. The model output is useful for:

- Brokers, to decide which claims need more evidence or manual review
- Insurers, to triage suspicious claims faster
- Claims teams, to reduce document-chasing and speed up processing
- Underwriters, to understand which policy and claim patterns are risky

For RiskProof AI, this tabular fraud score can later be combined with OCR, image, and audio signals to create a stronger multimodal claim-risk engine.

/home/sanjay/Documents/MLOps training/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([1.10948166])